In [0]:
!pip install dbt-core==1.8.4
!pip install dbt-databricks==1.8.4

In [0]:
import os
import shutil
from dbt.cli.main import dbtRunner


schema = "bronze"
host = "dbc-283c562e-fc2f.cloud.databricks.com"
http_path =  "/sql/1.0/warehouses/4f1a38c405c910cb"
databricks_token = dbutils.widgets.get("databricks_token")
catalog = "warehouse"

required = {
    "host": host,
    "http_path": http_path,
    "databricks_token": databricks_token,
    "catalog": catalog,
    "schema": schema,
    "artifact_folder": "artifact-dir"
}

In [0]:
DBT_PROJECT_DIR = "../../warehouse_dbt/"
LOCAL_ARTIFACT_DIR = "./dbt/artifact-dir"
LOCAL_STATE_DIR = os.path.join(DBT_PROJECT_DIR, "dbt-state")
PROFILES_DIR = os.path.expanduser("~/.dbt")
PROFILES_PATH = os.path.join(PROFILES_DIR, "profiles.yml")
artifact_dir_full_path = os.path.join(DBT_PROJECT_DIR, "artifact-dir")

# Clean up state directory before run
if os.path.exists(LOCAL_STATE_DIR):
    shutil.rmtree(LOCAL_STATE_DIR)
    print(f"Pre-cleaned existing state directory: {LOCAL_STATE_DIR}")

# Clean BEFORE build
if os.path.exists(artifact_dir_full_path):
    shutil.rmtree(artifact_dir_full_path)
    print(f"Pre-cleaned existing artifact directory: {artifact_dir_full_path}")

# Create yml profile
os.makedirs(PROFILES_DIR, exist_ok=True)
with open(PROFILES_PATH, "w") as f:
    f.write(
        f"""
warehouse_dbt:
    target: dev
    outputs:
        dev:
          type: databricks
          catalog: {catalog}
          schema: {schema}
          host: {host}
          http_path: {http_path}
          token: {databricks_token}
          threads: 2
        """
    )
print(f"Created profile file: {PROFILES_PATH}")

print("Running dbt deps...")
prev_dir = os.getcwd()
os.chdir(DBT_PROJECT_DIR)
try:
    dbt = dbtRunner()
    res_deps = dbt.invoke(["deps"])
    if not res_deps.success:
        raise RunTimeError("dbt deps failed.")
    print("dbt deps completed successfully")
finally:
    os.chdir(prev_dir)

# Run dbt build
print("Running dbt build...")
os.chdir(DBT_PROJECT_DIR)
try:
    res_build = dbt.invoke([
        "build",
        "--fail-fast",
        "--target-path", "artifact-dir"
    ])
    if not res_build.success:
        for node in res_build.result:
            print(f"Model {node.node.name}: {node.status}")
        raise RuntimeError("dbt build failed")
    print("dbt build completed successfully")
finally:
    os.chdir(prev_dir)

# Clean AFTER build
if os.path.exists(artifact_dir_full_path):
    shutil.rmtree(artifact_dir_full_path)
print(f"Post-cleaned artifact directory: {artifact_dir_full_path}")